In [1]:
!pip install torch-levenberg-marquardt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.5/963.5 kB 56.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-

# shulman :

In [ ]:
import time
import torch
import numpy as np
import pandas as pd
import torch_levenberg_marquardt as tlm
from torch.utils.data import TensorDataset
from bokeh.plotting import figure, show, output_notebook

# Set high matmul precision
torch.set_float32_matmul_precision('high')

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === Load your data from CSV ===
df = pd.read_csv("/content/magnesio-anthophyllite.csv")  # <-- update with your filename
T_data = torch.tensor(df["T"].values, dtype=torch.float32).unsqueeze(1).to(device)
Cp_data = torch.tensor(df["Cp"].values, dtype=torch.float32).unsqueeze(1).to(device)

# Create dataset and dataloader
dataset = TensorDataset(T_data, Cp_data)
loader = tlm.utils.FastDataLoader(dataset, batch_size=512, repeat=10, shuffle=True, device=device)

# === Define Cp model as a neural net with 6 trainable params ===
class CpModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.a = torch.nn.Parameter(torch.tensor([7.73e-3], dtype=torch.float32))
        self.b = torch.nn.Parameter(torch.tensor([1.263e-3], dtype=torch.float32))
        self.c = torch.nn.Parameter(torch.tensor([3.0], dtype=torch.float32))
        self.d = torch.nn.Parameter(torch.tensor([8.47e-3], dtype=torch.float32))
        self.e = torch.nn.Parameter(torch.tensor([2.0825e-7], dtype=torch.float32))
        self.f = torch.nn.Parameter(torch.tensor([4.97e-2], dtype=torch.float32))

    def forward(self, T):
        T = torch.clamp(T, min=1e-6)
        term1 = self.a * T
        term2 = 1 - torch.exp(-self.b * T**2)
        term3 = 1.0
        term6 = torch.exp(-self.c * torch.sqrt(T))
        term4 = self.d * T**6
        term5 = self.e * T**4 * torch.exp(-self.f * T)
        return term1 * term2 * (term3 + term6 * term4 + term5)

# Instantiate models
model_adam = CpModel().to(device)
model_lm = CpModel().to(device)

print(f"Number of parameters: {sum(p.numel() for p in model_adam.parameters())}")

# === Wrap with training modules ===
module_adam = tlm.training.OptimizerModule(
    model=model_adam,
    optimizer=torch.optim.Adam(model_adam.parameters(), lr=1e-2),
    loss_fn=torch.nn.MSELoss(),
)

module_lm = tlm.training.LevenbergMarquardtModule(
    model=model_lm,
    loss_fn=tlm.loss.MSELoss(),
    learning_rate=1.0,
    solve_method="qr",
    attempts_per_step=10,
)

# === Train using Adam ===
print("\nTraining with Adam...")
start = time.perf_counter()
tlm.utils.fit(module_adam, loader, epochs=10)
end = time.perf_counter()
print(f"Adam training took {end - start:.2f} seconds")

# === Train using Levenberg-Marquardt ===
print("\nTraining with Levenberg-Marquardt...")
start = time.perf_counter()
tlm.utils.fit(module_lm, loader, epochs=10)
end = time.perf_counter()
print(f"LM training took {end - start:.2f} seconds")

# === Evaluate and Plot ===
print("\nPlotting results...")
with torch.no_grad():
    y_adam = model_adam(T_data)
    y_lm = model_lm(T_data)

T_np = T_data.cpu().numpy().flatten()
Cp_np = Cp_data.cpu().numpy().flatten()
y_adam_np = y_adam.cpu().numpy().flatten()
y_lm_np = y_lm.cpu().numpy().flatten()

output_notebook()
p = figure(title="Cp(T) Fit: Adam vs Levenberg-Marquardt", width=800, height=400)
p.circle(T_np, Cp_np, size=3, color="black", legend_label="Data")
p.line(T_np, y_adam_np, color="green", line_dash="dashed", line_width=2, legend_label="Adam")
p.line(T_np, y_lm_np, color="red", line_width=2, legend_label="LM")
p.legend.location = "top_left"
p.xaxis.axis_label = "T"
p.yaxis.axis_label = "Cp"
show(p)


Using device: cpu
Number of parameters: 6

Training with Adam...


Epoch 10/10: 100%|██████████| 10/10 [00:00<00:00, 162.85it/s, epoch_avg_loss=nan]


Training complete. Final epoch average loss: nan
Adam training took 0.45 seconds

Training with Levenberg-Marquardt...


Epoch 7/10:  20%|██        | 2/10 [00:00<00:00, 105.77it/s, epoch_avg_loss=7.2030e-07]


Training complete. Final epoch average loss: 7.2030e-07
LM training took 0.42 seconds

Plotting results...


In [ ]:
def print_parameters(model):
    print("\nBest-fit parameters:")
    for name, param in model.named_parameters():
        print(f"{name} = {param.item():.8f}")

print_parameters(model_lm)



Best-fit parameters:
a = 0.00226886
b = 0.00028236
c = 4.48189735
d = 6.39312553
e = 0.00000001
f = 0.02022376
